In [1]:
import pandas as pd
import duckdb

data = [
    # Device A
    ["A", "2026-07-01 08:00:00", "NORMAL", 72],
    ["A", "2026-07-01 08:03:00", "ERROR", 88],
    ["A", "2026-07-01 08:05:00", "NORMAL", 76],
    ["A", "2026-07-01 08:08:00", "ERROR", 93],

    # Device B
    ["B", "2026-07-01 08:01:00", "NORMAL", 70],
    ["B", "2026-07-01 08:04:00", "ERROR", 91],
    ["B", "2026-07-01 08:06:00", "ERROR", 95],
    ["B", "2026-07-01 08:09:00", "NORMAL", 82],

    # Device C
    ["C", "2026-07-01 08:02:00", "NORMAL", 68],
    ["C", "2026-07-01 08:07:00", "NORMAL", 73],
    ["C", "2026-07-01 08:10:00", "ERROR", 89],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "collect_time", "status", "temp_value"]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)



   device_id        collect_time  status  temp_value
0          A 2026-07-01 08:00:00  NORMAL          72
1          A 2026-07-01 08:03:00   ERROR          88
2          A 2026-07-01 08:05:00  NORMAL          76
3          A 2026-07-01 08:08:00   ERROR          93
4          B 2026-07-01 08:01:00  NORMAL          70
5          B 2026-07-01 08:04:00   ERROR          91
6          B 2026-07-01 08:06:00   ERROR          95
7          B 2026-07-01 08:09:00  NORMAL          82
8          C 2026-07-01 08:02:00  NORMAL          68
9          C 2026-07-01 08:07:00  NORMAL          73
10         C 2026-07-01 08:10:00   ERROR          89


## 题目要求

### 分别使用 SQL 和 Pandas 完成：

- 找出每个设备中，temp_value 最高的前 2 条记录。

- **最终输出字段：**

`device_id`
`collect_time`
`status`
`temp_value`
`rn`

- **最终结果应该是：**

device_id | collect_time          | status | temp_value | rn|
|---------|-----------------------|--------|------------|---|
A         | 2026-07-01 08:08:00   | ERROR  | 93         | 1|
A         | 2026-07-01 08:03:00   | ERROR  | 88         | 2|
B         | 2026-07-01 08:06:00   | ERROR  | 95         | 1|
B         | 2026-07-01 08:04:00   | ERROR  | 91         | 2|
C         | 2026-07-01 08:10:00   | ERROR  | 89         | 1|
C         | 2026-07-01 08:07:00   | NORMAL | 73         | 2|

In [13]:
# SQL轨道

query = """
WITH rank_table AS (
SELECT
    device_id,
    collect_time,
    status,
    temp_value,
    ROW_NUMBER() OVER(PARTITION BY device_id ORDER BY temp_value DESC,collect_time DESC) AS rn
FROM df
)
SELECT 
    *
FROM rank_table
WHERE rn <= 2
ORDER BY device_id
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,collect_time,status,temp_value,rn
0,A,2026-07-01 08:08:00,ERROR,93,1
1,A,2026-07-01 08:03:00,ERROR,88,2
2,B,2026-07-01 08:06:00,ERROR,95,1
3,B,2026-07-01 08:04:00,ERROR,91,2
4,C,2026-07-01 08:10:00,ERROR,89,1
5,C,2026-07-01 08:07:00,NORMAL,73,2


In [15]:
# PANDAS轨道

df_pd = (
    df
    .sort_values(by=['device_id','temp_value','collect_time'],ascending=[True,False,False])
    .assign(
        rn = lambda x:x.groupby('device_id').cumcount() + 1
        )

    .loc[lambda x:x['rn'] <= 2]
    
    .reset_index(drop=True)
)   
df_pd

,device_id,collect_time,status,temp_value,rn
0,A,2026-07-01 08:08:00,ERROR,93,1
1,A,2026-07-01 08:03:00,ERROR,88,2
2,B,2026-07-01 08:06:00,ERROR,95,1
3,B,2026-07-01 08:04:00,ERROR,91,2
4,C,2026-07-01 08:10:00,ERROR,89,1
5,C,2026-07-01 08:07:00,NORMAL,73,2
